In [0]:
from pyspark.sql.functions import round, avg, min, max, count

# ── READ FROM SILVER ─────────────────────────────────
silver_df = spark.table("silver.exchange_rates")

# ── GOLD TABLE 1: Currency Summary by Category ───────
category_summary = silver_df.groupBy("rate_category") \
    .agg(
        count("currency").alias("total_currencies"),
        round(avg("rate"), 4).alias("avg_rate"),
        round(min("rate"), 4).alias("min_rate"),
        round(max("rate"), 4).alias("max_rate")
    ) \
    .orderBy("rate_category")

# ── GOLD TABLE 2: Top 20 Strongest Currencies vs USD ─
strongest_currencies = silver_df \
    .filter(silver_df.rate_category == "stronger_than_usd") \
    .orderBy("rate") \
    .limit(20) \
    .select("currency", "rate", "rate_vs_usd", "rate_category")

# ── GOLD TABLE 3: Top 20 Weakest Currencies vs USD ───
weakest_currencies = silver_df \
    .filter(silver_df.rate_category == "weaker_than_usd") \
    .orderBy("rate", ascending=False) \
    .limit(20) \
    .select("currency", "rate", "rate_vs_usd", "rate_category")

# ── SAVE TO GOLD ─────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

category_summary.writeTo("gold.category_summary") \
    .using("delta").createOrReplace()

strongest_currencies.writeTo("gold.strongest_currencies") \
    .using("delta").createOrReplace()

weakest_currencies.writeTo("gold.weakest_currencies") \
    .using("delta").createOrReplace()

print("Gold tables saved successfully")

# ── PREVIEW ──────────────────────────────────────────
print("\n── Category Summary ──")
spark.sql("SELECT * FROM gold.category_summary").show()

print("\n── Top 20 Strongest Currencies ──")
spark.sql("SELECT * FROM gold.strongest_currencies").show()

print("\n── Top 20 Weakest Currencies ──")
spark.sql("SELECT * FROM gold.weakest_currencies").show()